# Testing your model {#sec-testing}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c09-testing.ipynb)

A model that runs is not yet a model you can trust. Before anyone uses the waiting-list model to argue for more clinics, two questions need answers. Does the code do what the equations say? And is the model fit for the question it is meant to answer? This chapter turns both questions into tests you can run, read and repeat, and ends with a test sheet that checks the model every time it changes.

**Who it is for.** Anyone who will hand a model's results to someone else. **Who can skip.** No one should skip it completely; experienced modellers can skim sections 2 to 4 and go to the test sheet in section 8. **Prerequisites.** Stocks and flows (chapter 1, Why systems push back), the ageing chain (chapter 4, Delays, pipelines and memory), and protecting a stock from going negative (chapter 5, Decisions inside the model). **Time.** About three hours with the exercises.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 8
rng = np.random.default_rng(SEED)

DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/main/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])

# Okabe-Ito colours (safe for colour-blind readers) with a distinct line style
# for each, so every figure also reads in greyscale
plt.rcParams["axes.prop_cycle"] = (
    cycler(color=["#000000", "#E69F00", "#56B4E9", "#009E73",
                  "#0072B2", "#D55E00", "#CC79A7"])
    + cycler(linestyle=["-", "--", ":", "-.", (0, (5, 1)),
                        (0, (3, 1, 1, 1)), (0, (1, 1))]))
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3})
print("loopbricks", lb.__version__, "| data:", len(wl), "months")

In [ ]:
from matplotlib.patches import FancyArrowPatch, Rectangle


def draw_case(ax):
    """Stock-and-flow sketch of the waiting list in four bands."""
    pipe = dict(arrowstyle="-|>", mutation_scale=15, lw=3, color="0.3")
    link = dict(arrowstyle="-|>", mutation_scale=10, lw=1, color="0.3",
                linestyle="--")
    names = ["0 to 3", "3 to 6", "6 to 12", "12 plus"]
    xs = [0.12, 0.34, 0.56, 0.78]
    for x, n in zip(xs, names):
        ax.add_patch(Rectangle((x, 0.48), 0.14, 0.2, fill=False, lw=2))
        ax.text(x + 0.07, 0.58, f"Waiting\n{n}", ha="center",
                va="center", fontsize=8)
        ax.add_patch(FancyArrowPatch((x + 0.07, 0.48), (x + 0.07, 0.2),
                                     **pipe))
    for a, b in zip(xs[:-1], xs[1:]):
        ax.add_patch(FancyArrowPatch((a + 0.14, 0.58), (b, 0.58), **pipe))
        ax.text((a + 0.14 + b) / 2, 0.62, "ageing", ha="center",
                fontsize=7)
    ax.add_patch(FancyArrowPatch((0.0, 0.58), (0.12, 0.58), **pipe))
    ax.text(0.04, 0.63, "Referrals", ha="center", fontsize=8)
    ax.text(0.5, 0.12, "Treating 0 to 3 ... Treating 12 plus: shares of "
            "Treatments = MIN(Capacity, list / minimum treatment time)",
            ha="center", fontsize=8)
    ax.text(0.5, 0.9, "Capacity: trend x 2020 shock x 2022 initiative",
            ha="center", fontsize=8)
    ax.add_patch(FancyArrowPatch((0.5, 0.86), (0.5, 0.34), **link))
    ax.set_xlim(-0.02, 0.96)
    ax.set_ylim(0.05, 0.95)
    ax.axis("off")


fig, ax = plt.subplots(figsize=(8, 3.2))
draw_case(ax)
ax.set_title("The waiting-list model under test")
plt.show()

Figure: the waiting-list model tested in this chapter. Referrals enter the first band, patients age from band to band, and capacity is shared out as treatments across all four bands. Alt text: four boxes in a row joined by thick arrows, an inflow on the left and an outflow from each box.

**Learning outcomes.** By the end of the chapter you will be able to

1. tell verification from validation and name a test for each (Try it 1);
2. build a unit checker and a mass-balance check and use them to catch planted errors (Try its 2 and 3);
3. diagnose time-step and integration error by halving `dt` and comparing Euler with fourth-order Runge-Kutta (Try it 4);
4. design extreme-condition, plausibility and boundary tests and justify a fix when one fails (Try it 5);
5. use behaviour-pattern measures to compare a model with history (Try it 6);
6. run one-at-a-time and Monte Carlo sensitivity analyses and say which conclusions they threaten (Try it 7);
7. write a reusable test sheet that runs by itself (Try it 8).

## 1. Two questions: is the code right, and is the model right?

**Verification** asks whether the program computes what the equations say. A misspelt name, a forgotten unit conversion or a time step that is too large are verification failures. They have right answers: the code is either faithful to the equations or it is not.

**Validation** asks whether the model is good enough for its purpose. No model is true. A model earns trust by passing tests it could have failed, and the tests that matter depend on the question it must answer [@barlas1996]. Barlas separates tests of **structure** (are the equations and parameters defensible, one by one and under extreme conditions?) from tests of **behaviour** (does the model produce the patterns we see?). He argues that structure comes first: a model can fit the history for the wrong reasons. Martinez-Moyano and Richardson's survey of experienced modellers puts testing throughout the work, not at the end [@martinezmoyano2013].

The table maps this chapter's tests onto those two questions.

| Question | Test | Section |
|---|---|---|
| Verification | units agree in every equation | 2 |
| Verification | every patient adds up (mass balance) | 3 |
| Verification | results do not depend on `dt` or the integration method | 4 |
| Validation, structure | extreme conditions, parameter plausibility, boundary adequacy | 5 |
| Validation, behaviour | pattern measures against the reference mode | 6 |
| Both | sensitivity: which assumptions carry the answer | 7 |

Goodness of fit, calibration and uncertainty about parameters are the subject of chapter 10 (Data, calibration and uncertainty).

The model under test is the running case with the list split into four bands of time waited, as in chapter 4. Treatments are shared across the bands in proportion to their size, with the shortest waits weighted up by a `Priority` factor for urgent cases. Capacity grows slowly, collapses in March 2020 and recovers, and gets a 10% boost from a waiting-list initiative in January 2022. The values below were set by eye, close enough to the data to be useful; chapter 10 estimates them properly. The data are synthetic, shaped like published Irish waiting-list counts; see chapter 10.

In [ ]:
BANDS = ["Waiting 0 to 3", "Waiting 3 to 6", "Waiting 6 to 12",
         "Waiting 12 plus"]
first = wl.iloc[0]


def waiting_list_spec(guard=True, feedback=False):
    """The running case as four bands of time waited (an ageing chain).

    TIME counts months from the first count, at the end of January 2014.
    guard=False leaves out the protection against an empty list.
    feedback=True adds referrals that fall when the wait gets longer.
    """
    def var(name, eq, units):
        return {"id": lb.name_key(name), "name": name, "eq": str(eq),
                "units": units}

    def stock(i, name, init):
        return {"id": f"b{i}", "name": name, "init": str(init),
                "units": "patients", "nonNegative": True}

    def flow(name, eq, frm=None, to=None):
        f = {"id": lb.name_key(name), "name": name, "eq": eq,
             "units": "patients/month"}
        if frm:
            f["from"] = frm
        if to:
            f["to"] = to
        return f

    weighted = ("Priority * Waiting_0_to_3 + Waiting_3_to_6"
                " + Waiting_6_to_12 + Waiting_12_plus")
    if guard:
        weighted = f"MAX({weighted}, Smallest_list)"
    referrals = ("Referrals_0 * EXP(Referral_growth * TIME"
                 " / Months_per_year)")
    if feedback:
        referrals += " * (Expected_wait / Reference_wait) ^ (-Wait_elasticity)"
    stocks = [stock(0, BANDS[0], first.w0_3), stock(1, BANDS[1], first.w3_6),
              stock(2, BANDS[2], first.w6_12),
              stock(3, BANDS[3], first.w12_plus)]
    flows = [
        flow("Referrals", referrals, to="b0"),
        flow("Ageing 0", "Waiting_0_to_3 / Band_time_0", "b0", "b1"),
        flow("Ageing 1", "Waiting_3_to_6 / Band_time_1", "b1", "b2"),
        flow("Ageing 2", "Waiting_6_to_12 / Band_time_2", "b2", "b3"),
        flow("Treating 0",
             "Treatments * Priority * Waiting_0_to_3 / Weighted_list", "b0"),
        flow("Treating 1", "Treatments * Waiting_3_to_6 / Weighted_list",
             "b1"),
        flow("Treating 2", "Treatments * Waiting_6_to_12 / Weighted_list",
             "b2"),
        flow("Treating 3", "Treatments * Waiting_12_plus / Weighted_list",
             "b3"),
    ]
    variables = [
        var("Waiting list", "Waiting_0_to_3 + Waiting_3_to_6"
            " + Waiting_6_to_12 + Waiting_12_plus", "patients"),
        var("Weighted list", weighted, "patients"),
        var("Treatments",
            "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
            "patients/month"),
        var("Capacity", "Capacity_0 * EXP(Capacity_growth * TIME"
            " / Months_per_year) * Shock_effect"
            " * (1 + STEP(Initiative_boost, Initiative_start))",
            "patients/month"),
        var("Shock effect", "1 - STEP(1, Shock_start)"
            " * (1 - Activity_at_worst)"
            " * MAX(0, 1 - (TIME - Shock_start) / Recovery_time)", "dmnl"),
        var("Expected wait", "Waiting_list / MAX(Treatments, Smallest_rate)",
            "month"),
        var("Referrals 0", 4000, "patients/month"),
        var("Referral growth", 0.03, "1/year"),
        var("Capacity 0", 3950, "patients/month"),
        var("Capacity growth", 0.02, "1/year"),
        var("Activity at worst", 0.35, "dmnl"),
        var("Recovery time", 9, "month"),
        var("Shock start", 73, "month"),
        var("Initiative boost", 0.10, "dmnl"),
        var("Initiative start", 95, "month"),
        var("Priority", 1.4, "dmnl"),
        var("Minimum treatment time", 0.5, "month"),
        var("Months per year", 12, "month/year"),
        var("Band time 0", 3, "month"),
        var("Band time 1", 3, "month"),
        var("Band time 2", 6, "month"),
        var("Smallest list", 1, "patients"),
        var("Smallest rate", 1, "patients/month"),
        var("Reference wait", 8, "month"),
        var("Wait elasticity", 0.3, "dmnl"),
    ]
    return {"name": "Waiting list by band", "timeUnit": "month", "T": 119,
            "dt": 0.25, "stocks": stocks, "flows": flows, "vars": variables}


case = lb.Model.from_spec(waiting_list_spec())
print(case.equations())

Every quantity carries a unit in the spec, which the next section puts to work. `Months per year` (month/year) converts the yearly growth rates, and `Smallest list` (patients) keeps a division safe when the list is empty. Here is one run against the data.

In [ ]:
def months(df):
    """The model's monthly values (TIME 0, 1, ..., 119) on the data's dates."""
    out = df.loc[np.arange(len(wl), dtype=float)]
    return out.set_index(wl["month"])


base = case.run()
fig, ax = plt.subplots()
ax.plot(wl["month"], wl["total"] / 1000, label="data (synthetic)")
ax.plot(wl["month"], months(base)["Waiting list"] / 1000,
        label="model, values set by eye")
ax.set(title="The waiting list, data and first model",
       xlabel="month", ylabel="patients waiting (thousands)")
ax.legend()
for d in ["2014-01", "2019-12", "2021-12", "2023-12"]:
    i = int(np.flatnonzero(wl["month"] == d)[0])
    print(f"{d}: data {wl.total[i]:6,}  model "
          f"{base['Waiting list'].iloc[4 * i]:8,.0f} patients")
plt.show()

Figure: the synthetic waiting list (solid) and the model with values set by eye (dashed), January 2014 to December 2023, in thousands of patients. Alt text: two rising lines that stay close, with a sharp rise in 2020 and a flat stretch after 2022.

The model ends 2023 at 68,836 patients against 69,927 in the data, and is 1,449 patients high at the end of 2019. Close is not the same as trustworthy, so on to the tests.

**Try it 1.** *Check:* is "the model's December 2023 list is within 2% of the data" a verification or a validation test? *Apply:* a colleague finds that the model gives a different answer on a different laptop. Which kind of failure is that? *Extend:* name one structure test that could fail even when the fit to history is perfect.

<details><summary>Worked answer</summary>

*Check:* validation, of behaviour. It compares the model with the world, not the code with the equations. *Apply:* verification. The equations are the same on both machines, so the arithmetic, a library version or a hidden random seed differs. *Extend:* an extreme-condition test, such as setting capacity to zero. A model can match ten years of data and still let the list go negative, or crash, when pushed outside that history; section 5 finds both faults in this model.
</details>

## 2. Verification: units agree in every equation

An equation that adds patients to patients per month is wrong whatever numbers go into it. **Dimensional consistency** means that the two sides of every equation, and both sides of every `+`, `-`, `MIN` and `MAX`, carry the same unit. Checking by eye is slow and unreliable, so we let Python do it.

The idea is short. Represent a unit as powers of base units, so patients/month is `{"patients": 1, "month": -1}`. Multiplying units adds the powers, dividing subtracts them, and adding demands equal powers. Then evaluate each equation with units in place of numbers. `EXP` and `LN` need an argument without a unit, and the start time of a `STEP` must be a time.

In [ ]:
class UnitError(Exception):
    pass


class Unit:
    """A unit as powers of base units.

    patients/month is {"patients": 1, "month": -1}; dmnl (no unit) is {}.
    """

    def __init__(self, powers=None):
        self.p = {k: v for k, v in (powers or {}).items() if v}

    @classmethod
    def parse(cls, text):
        if text in ("", "dmnl", "1"):
            return cls()
        top, *below = text.split("/")
        powers = {}
        for part, sign in [(top, 1)] + [(b, -1) for b in below]:
            for name in part.split("*"):
                name = name.strip()
                if name not in ("", "1"):
                    powers[name] = powers.get(name, 0) + sign
        return cls(powers)

    def __repr__(self):
        if not self.p:
            return "dmnl"
        top = "*".join(k for k, v in self.p.items() for _ in range(v)) or "1"
        low = "*".join(k for k, v in self.p.items() for _ in range(-v))
        return top + ("/" + low if low else "")

    def __mul__(self, other):
        o = as_unit(other)
        keys = set(self.p) | set(o.p)
        return Unit({k: self.p.get(k, 0) + o.p.get(k, 0) for k in keys})

    __rmul__ = __mul__

    def __truediv__(self, other):
        return self * as_unit(other) ** -1

    def __rtruediv__(self, other):
        return as_unit(other) / self

    def __pow__(self, n):
        if isinstance(n, Unit):
            if n.p:
                raise UnitError(f"an exponent in {n}")
            if self.p:
                raise UnitError(f"{self} to a power the model works out")
            return self
        return Unit({k: v * n for k, v in self.p.items()})

    def __add__(self, other):
        o = as_unit(other)
        if o.p != self.p:
            raise UnitError(f"adds or subtracts {self} and {o}")
        return self

    __radd__ = __sub__ = __rsub__ = __add__

    def __neg__(self):
        return self


def as_unit(x):
    return x if isinstance(x, Unit) else Unit()   # a plain number has no unit


def same_units(*args):
    """MIN and MAX: every argument in the same unit (a bare 0 fits any)."""
    units = [as_unit(a) for a in args if isinstance(a, Unit) or a != 0]
    for u in units[1:]:
        if u.p != units[0].p:
            raise UnitError(f"compares {units[0]} with {u}")
    return units[0] if units else Unit()


def no_units(x):
    if as_unit(x).p:
        raise UnitError(f"takes EXP or LN of {x}")
    return Unit()


def check_units(model):
    """Check every flow and variable against its declared unit.

    Returns a list of problems, empty when every equation agrees.
    """
    env = {lb.name_key(e["name"]): Unit.parse(e.get("units", ""))
           for e in model.stocks + model.flows + model.vars}
    t_unit = Unit.parse(model.time_unit)

    def step(height, start):
        as_unit(start) + t_unit           # the start must be a time
        return height

    funcs = {"MIN": same_units, "MAX": same_units, "EXP": no_units,
             "LN": no_units, "ABS": lambda x: x, "STEP": step,
             "TIME": t_unit}
    problems = []
    for e in model.flows + model.vars:
        try:
            float(e["eq"])
            continue                      # a constant: its unit is declared
        except ValueError:
            pass
        code = re.sub(r"(?<![\w.])[A-Za-z_]\w*",
                      lambda m: m.group() if m.group() in funcs
                      else f'env["{lb.name_key(m.group())}"]', e["eq"])
        try:
            got = as_unit(eval(code.replace("^", "**"), {"env": env, **funcs}))
        except UnitError as err:
            problems.append(f"{e['name']}: {err}")
            continue
        want = env[lb.name_key(e["name"])]
        if got.p != want.p:
            problems.append(f"{e['name']}: equation gives {got}, "
                            f"declared {want}")
    stock_units = {s["id"]: Unit.parse(s["units"]) for s in model.stocks}
    for f in model.flows:
        for end in ("from", "to"):
            if f.get(end):
                rate = stock_units[f[end]] / t_unit
                if rate.p != Unit.parse(f["units"]).p:
                    problems.append(f"{f['name']}: {f['units']} cannot "
                                    f"change a stock in {stock_units[f[end]]}")
    return problems


print("case model:", check_units(case) or "no unit problems")
draft = case.with_equation("Treatments", "MIN(Capacity, Waiting_list)")
print("draft:", check_units(draft))

The case model passes: every equation agrees with its declared units. The draft, in which someone forgot to divide the list by the minimum treatment time, fails at once, with a message that names the equation and the clash (patients/month against patients). A run of the draft would have gone ahead without complaint and produced numbers.

The checker also explains two choices in the model. Growth is written as `EXP(Referral_growth * TIME / Months_per_year)`, because `Referral_growth` is per year and `TIME` is in months; the named constant `Months per year` (month/year) makes the exponent unitless. And the start of the shock is a named constant, `Shock start` (month), rather than a bare 73.

**Try it 2.** *Check:* remove `/ Months_per_year` from `Referrals` and run the checker. *Apply:* a colleague multiplies `Treatments` by `Months_per_year` to "convert it to a yearly figure". What does the checker say? *Extend:* write the initiative as `STEP(Initiative_boost, 95)`, with a bare number for the start. Does the checker accept it, and is that the behaviour you want?

<details><summary>Worked answer</summary>

The next cell runs all three. *Check:* `Referrals: takes EXP or LN of month/year`; the exponent is no longer a pure number. *Apply:* `Treatments: equation gives patients/year, declared patients/month`. The model would treat twelve times too many patients each month. *Extend:* `Capacity: adds or subtracts dmnl and month`. The bare 95 has no unit, so it cannot be compared with `TIME`. That is strict but useful: it pushes every time constant into a named, documented parameter, which is one of the habits recommended by @martinezmoyano2013.
</details>

In [ ]:
slip = case.with_equation(
    "Referrals", "Referrals_0 * EXP(Referral_growth * TIME)")
print("check: ", check_units(slip))
yearly = case.with_equation(
    "Treatments",
    "MIN(Capacity, Waiting_list / Minimum_treatment_time) * Months_per_year")
print("apply: ", check_units(yearly))
bare = case.with_equation(
    "Capacity", "Capacity_0 * EXP(Capacity_growth * TIME / Months_per_year)"
    " * Shock_effect * (1 + STEP(Initiative_boost, 95))")
print("extend:", check_units(bare))

## 3. Verification: every patient adds up

A stock is the sum of what flowed in minus what flowed out. With Euler's method (chapter 1), the stock at each step is exactly its starting value plus `dt` times the net flow of every earlier step. So a **mass-balance check** adds the flows up independently and compares them with the stock. Any gap beyond rounding error means the simulator changed the stock some other way. The usual culprit is the **floor at zero**: when an outflow asks for more than a stock holds, the stock is held at zero, and the books no longer balance. The package records the first time this happens in `df.attrs["clipped"]`.

In [ ]:
def balance_error(model, **run_args):
    """Largest gap, per stock, between the stock and its flows added up."""
    df = model.run(include="all", **run_args)
    dt = run_args.get("dt", model.dt)
    gaps = {}
    for s in model.stocks:
        ins = [f["name"] for f in model.flows if f.get("to") == s["id"]]
        outs = [f["name"] for f in model.flows if f.get("from") == s["id"]]
        net = df[ins].sum(axis=1) - df[outs].sum(axis=1)
        added_up = (df[s["name"]].iloc[0]
                    + dt * net.cumsum().shift(fill_value=0))
        gaps[s["name"]] = float((df[s["name"]] - added_up).abs().max())
    return gaps, df.attrs["clipped"]


gaps, clipped = balance_error(case)
print({k: f"{v:.1e}" for k, v in gaps.items()}, "clipped:", clipped)
df = case.run(include="all")
treated = df[[f"Treating {i}" for i in range(4)]].sum(axis=1)
added_up = (df["Waiting list"].iloc[0]
            + 0.25 * (df["Referrals"] - treated).cumsum().shift(fill_value=0))
gap = (df["Waiting list"] - added_up).abs().max()
print(f"whole list: largest gap {gap:.1e} patients")

The largest gap in any band is about $10^{-11}$ patients: rounding error and nothing else. The second check is the important one for a chain. Ageing flows move patients between bands, so they must cancel when we add the bands together. The whole list equals its starting value plus referrals minus treatments, to $6 \times 10^{-11}$ patients. A cascade that conserves its material passes this test; one that loses or creates patients between stages fails it.

### Four molecules that test the floor

Four molecules from Hines' collection make the point sharply, and we use them as test subjects. Each is a few lines, and each either passes or fails the mass-balance check for a reason you can see in its equations.

**Reducing Backlog** (`reducingBacklog`) drains a stock of work by workers times productivity, after Hines, *Molecules of Structure* [@hines_molecules]. It is the simplest way to model work getting done, and it has a flaw on purpose: the outflow does not look at the stock.

$$\frac{dW}{dt} = -P, \qquad P = N \times p, \qquad \text{Overdrawing} = P - \min\!\left(P, \frac{W}{\Delta t}\right)$$

Here $W$ is `Work to do` (tasks), $N$ is `Workers` (people) and $p$ is `Productivity` (tasks per person per month). The `Overdrawn` stock is our addition: a meter that counts work taken from an empty stock, so the flaw can be measured.

**Level Protected by PDY** (`protByPDY`), after Hines, *Molecules of Structure* [@hines_molecules], fixes the flaw by letting productivity fall as the work runs thin. When less than `Work for full PDY` (tasks) remains, each worker finds less to do:

$$\text{Producing} = N \, p_n \min\!\left(1, \frac{R}{R^*}\right)$$

with $R$ the `Remaining work`, $R^*$ the `Work for full PDY` and $p_n$ the `Normal PDY`. Ours guards the division with a tiny lower bound on $R^*$. This is a **floor that binds**: the protection is part of the equations, so the floor at zero in the simulator never has to act.

In [ ]:
rb = lb.molecule("reducingBacklog")
pp = lb.molecule("protByPDY")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    run_rb = rb.run(preset="Straight-line decline")
run_pp = pp.run(preset="Run the backlog down")
print("reducingBacklog clipped:", run_rb.attrs["clipped"])
print(run_rb.loc[[0, 29, 30, 31, 60], ["Work to do", "Producing",
                                        "Overdrawn"]])
print("protByPDY clipped:", run_pp.attrs["clipped"])
print(run_pp.loc[[0, 1, 3, 6, 12, 60], ["Remaining work", "Producing",
                                         "Productivity"]].round(3))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.plot(run_rb.index, run_rb["Work to do"], label="Work to do")
a1.plot(run_rb.index, run_rb["Overdrawn"], label="Overdrawn")
a1.set(title="Reducing Backlog: work done from nothing",
       xlabel="time (month)", ylabel="tasks")
a1.legend()
a2.plot(run_pp.index, run_pp["Remaining work"], label="Remaining work")
a2.set(title="Level Protected by PDY: a floor that binds",
       xlabel="time (month)", ylabel="tasks")
a2.legend()
plt.tight_layout()
plt.show()

Figure: left, Reducing Backlog with 10 workers at 4 tasks a month; work to do falls in a straight line to zero at month 30, and the overdrawn meter then climbs. Right, Level Protected by PDY with 10 workers at a normal 5 tasks a month; remaining work falls in a straight line, then bends and approaches zero. Alt text: two panels, one with a falling line meeting a rising line, one with a falling line that flattens near zero.

Reducing Backlog empties its 1,200 tasks at month 30, the floor holds `Work to do` from month 30.1, and by month 60 the model has "produced" another 1,200 tasks from nothing. The clip report says so without anyone looking at a chart. Level Protected by PDY takes 50 tasks a month until month 4.8, when 60 tasks remain, and then slows: at month 6, 21.1 tasks remain and productivity has fallen from 5 to 1.76 tasks per person per month. Nothing is clipped, ever.

**Doing Work Cascade** (`doingWorkCascade`), after Hines, *Molecules of Structure* [@hines_molecules], chains three stocks with a crew at each stage. Every flow is a crew working at a fixed productivity, so each stage pulls from the stock behind it whether or not anything is there:

$$F_A = N_A\,p, \quad F_B = N_B\,p, \quad \frac{dA}{dt} = F_A - F_B, \quad \frac{dB}{dt} = F_B - F_C, \;\ldots$$

**Cascade Protected by PDY** (`cascadeProtByPDY`), after Hines, *Molecules of Structure* [@hines_molecules], applies the PDY protection at every stage, so each crew slows as the stock it draws from runs low:

$$F_B = N_B\,p_n \min\!\left(1, \frac{B}{R^*}\right), \quad F_C = N_C\,p_n \min\!\left(1, \frac{C}{R^*}\right), \;\ldots$$

In ours every stock has a floor at zero, so a failure shows up in the clip report as well as in the balance. Run the mass-balance check on three cases.

In [ ]:
dwc = lb.molecule("doingWorkCascade")
cpp = lb.molecule("cascadeProtByPDY")
for model, preset in [(dwc, "Balanced line"),
                      (dwc, "Starve stage A (the flaw)"),
                      (cpp, "Feed stops")]:
    gaps, clipped = balance_error(model, preset=preset, warn_clips=False)
    print(f"{model.name}, {preset}: largest gap {max(gaps.values()):.2f}"
          f" tasks; clipped {clipped}")

The balanced line conserves its tasks exactly. Starve stage A (2 workers feeding it, 8 drawing from it) and the floor holds `Inventory A` from month 8.4, after which the books drift by up to 620 tasks. The protected cascade, with its feed cut off, conserves everything: its crews simply slow down.

In [ ]:
starved = dwc.run(preset="Starve stage A (the flaw)", warn_clips=False)
stocks = ["Inventory A", "Inventory B", "Inventory C"]
in_chain = starved[stocks].sum(axis=1)
flows = starved["Producing A"] - starved["Producing D"]
books = in_chain.iloc[0] + 0.1 * flows.cumsum().shift(fill_value=0)
fed = cpp.run(preset="Feed stops")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.plot(starved.index, in_chain, label="in the chain")
a1.plot(starved.index, books, label="flows added up")
a1.set(title="Doing Work Cascade, stage A starved",
       xlabel="time (month)", ylabel="tasks")
a1.legend()
for s in ["Inventory B", "Inventory C", "Inventory D"]:
    a2.plot(fed.index, fed[s], label=s)
a2.set(title="Cascade Protected by PDY, feed stops",
       xlabel="time (month)", ylabel="tasks")
a2.legend()
plt.tight_layout()
print(f"month 60: chain holds {in_chain.iloc[-1]:.0f}, books say "
      f"{books.iloc[-1]:.0f} tasks")
print(fed.loc[[0, 12, 24, 60], ["Inventory B", "Inventory C", "Inventory D",
                                 "Producing D"]].round(2))
plt.show()

Figure: left, the starved Doing Work Cascade. The tasks actually in the chain (solid) part company with the flows added up (dashed) once stage A runs dry. Right, Cascade Protected by PDY after its feed stops: each inventory drains smoothly towards zero. Alt text: left panel with two lines that diverge after month 8; right panel with three falling curves.

By month 60 the starved chain holds 560 tasks while its flows say it should hold minus 60. A stock cannot hold negative tasks, which is the clearest possible sign that the books are being kept by the simulator's floor rather than by the model. In the protected cascade, stage D's output falls from 32 to 6.67 tasks a month by month 12 as its supply thins, and every stock approaches zero from above.

The lesson for the waiting list: our treatment flows look at the bands they draw from (through the shares and the minimum treatment time), which is why the case model passed. Section 5 finds a corner where that protection is not enough.

**Try it 3.** *Check:* run the mass-balance check on Doing Work Cascade with the preset "Unbalanced crews". When is a stock first clipped, and how big is the largest gap? *Apply:* protect only stage B of the starved cascade, with `Producing B = Workers_B * Productivity * MIN(1, Inventory_A / 20)`, and run the check again. *Extend:* in Level Protected by PDY, how long does it take to get below one task with `Work for full PDY` at 60 tasks ("Run the backlog down") and at 150 ("Early taper")?

<details><summary>Worked answer</summary>

The next cell does all three. *Check:* `Inventory A` is held at zero from month 50, and the gap reaches 20 tasks. *Apply:* stage A is now safe, but the trouble moves downstream: `Inventory B` is clipped from month 28.8 and the gap is 125 tasks, because stage C still pulls from B without looking. Protecting one stage is not enough; every flow that draws from a stock needs the protection, which is exactly what Cascade Protected by PDY does. *Extend:* 9.6 months with 60 tasks and 17.8 months with 150. A larger `Work for full PDY` starts the slowdown earlier, so the last tasks take longer: the protection has a price in speed, which is a modelling choice to defend, not a free fix.
</details>

In [ ]:
gaps, clipped = balance_error(dwc, preset="Unbalanced crews",
                              warn_clips=False)
print(f"check: largest gap {max(gaps.values()):.1f} tasks; "
      f"clipped {clipped}")
fixed_b = dwc.with_equation(
    "Producing B", "Workers_B * Productivity * MIN(1, Inventory_A / 20)")
gaps, clipped = balance_error(fixed_b, preset="Starve stage A (the flaw)",
                              warn_clips=False)
print(f"apply: largest gap {max(gaps.values()):.1f} tasks; "
      f"clipped {clipped}")
for preset in ["Run the backlog down", "Early taper"]:
    r = pp.run(preset=preset)
    below = r.index[r["Remaining work"] < 1][0]
    print(f"extend, {preset}: below one task from month {below}")

## 4. Verification: the arithmetic of time

A simulator steps through time in jumps of `dt`, holding every flow fixed within a step. The answer should belong to the model, not to the step. The test is to **halve `dt`** and see whether the answer moves by more than you would care about. A second test changes the **integration method**. Euler's method uses the flows at the start of each step. Fourth-order **Runge-Kutta** (`method="rk4"`) looks at the flows four times within each step and is far more accurate for smooth models. We run both on the case model as set, and on an extreme case with three times the capacity, where the list drains quickly.

In [ ]:
rows = []
for dt in [1, 0.5, 0.25, 0.125, 0.0625]:
    for method in ["euler", "rk4"]:
        for label, extra in [("as fitted", {}), ("capacity x 3",
                                                 {"capacity_0": 11850})]:
            r = case.run(dt=dt, method=method, include=["Waiting list"],
                         warn_clips=False, **extra)
            rows.append({"dt (month)": dt, "method": method, "case": label,
                         "list at month 119": r["Waiting list"].iloc[-1],
                         "clipped": ", ".join(r.attrs["clipped"]) or "-"})
steps = pd.DataFrame(rows)
fmt = {"list at month 119": "{:,.0f}".format, "dt (month)": "{:g}".format}
for label in ["as fitted", "capacity x 3"]:
    print(steps[steps.case == label].to_string(index=False, formatters=fmt))

Two findings, one for each table.

First, under normal conditions Euler's answer for December 2023 falls from 69,735 to 69,136, 68,836, 68,686 and 68,611 as `dt` halves. Each halving cuts the change by half (599, 300, 150, 75 patients): the signature of a first-order method. Extrapolating, the exact answer is near 68,535, so the model's `dt` of 0.25 months is about 300 patients (0.4%) high. That is small against the month-to-month noise in the data, so `dt = 0.25` is fit for this purpose.

Second, Runge-Kutta does not escape the pattern here. Its answers also move by halves (203, 102, 51 and 25 patients). Fourth-order accuracy needs smooth flows, and this model has jumps: the shock starts with a `STEP`, recovery ends at a kink, and the initiative is another `STEP`. A Runge-Kutta step that straddles a jump samples it part way and makes an error proportional to `dt`. Try it 4 shows that before 2020, where nothing jumps, RK4 is exact to a tenth of a patient even with `dt = 1`.

With three times the capacity, Euler with `dt = 1` clips all four bands. The minimum treatment time is 0.5 months, so the outflow can reach twice the list per month, and a one-month step removes more patients than there are. Every other run gives 2,690 patients. The clip report caught a time-step failure that a glance at the final number might have missed. Euler steps must be shorter than the shortest time constant in play (here the 0.5-month minimum treatment time) or they overshoot; a quarter to a half of it is a common starting point, and halving then confirms the choice.

**Try it 4.** *Check:* in the capacity-times-three table, what is the largest Euler `dt` that does not clip? *Apply:* run the model to month 72 only (December 2019, before any jump) with `dt` of 1, 0.5, 0.25 and 0.125, using Euler and RK4. *Extend:* use the four Euler differences above to estimate the order of Euler's error.

<details><summary>Worked answer</summary>

*Check:* 0.5 months, one step per minimum treatment time; 1 month clips. *Apply:* the next cell prints Euler 45,513.7, 45,585.0, 45,620.6 and 45,638.4 against RK4 45,656.2 at every step size. Without jumps, RK4 at `dt = 1` beats Euler at `dt = 0.125`. *Extend:* each difference is 2.00 times the next, so the error halves when `dt` halves: error proportional to `dt`, first order.
</details>

In [ ]:
for method in ["euler", "rk4"]:
    ends = [case.run(T=72, dt=dt, method=method,
                     include=["Waiting list"])["Waiting list"].iloc[-1]
            for dt in [1, 0.5, 0.25, 0.125]]
    print(method, [round(float(e), 1) for e in ends])
euler = steps[(steps.case == "as fitted") & (steps.method == "euler")]
d = np.diff(euler["list at month 119"].to_numpy())
print("differences:", d.round(1), "ratios:", (d[:-1] / d[1:]).round(2))

## 5. Validation: tests of structure

Structure tests ask whether each part of the model makes sense on its own terms, before any comparison with data [@barlas1996]. We run three kinds.

**Extreme conditions.** Push an input to an absurd value and ask whether the model still does something that could happen. No capacity: the list should grow by exactly the referrals. No referrals: the list should empty without going negative. Huge capacity: no band should be overdrawn. An empty list: the model should simply sit at zero.

In [ ]:
def extreme(label, model=case, **params):
    """Run one extreme case; report the list, the stocks and any clipping."""
    try:
        df = model.run(warn_clips=False, **params)
    except lb.LoopBricksError as err:
        return {"test": label, "result": "crashed", "detail": str(err)[:70]}
    low = df[BANDS].min().min()
    held = list(df.attrs["clipped"]) or "none"
    return {"test": label, "result": "ran",
            "detail": f"list {df['Waiting list'].iloc[0]:,.0f} to "
                      f"{df['Waiting list'].iloc[-1]:,.0f}; lowest band "
                      f"{low:,.1f}; clipped {held}"}


unguarded = lb.Model.from_spec(waiting_list_spec(guard=False))
tests = [
    extreme("no capacity", capacity_0=0),
    extreme("no referrals", referrals_0=0),
    extreme("ten times the capacity", capacity_0=39500),
    extreme("urgent first, ten times the capacity", priority=50,
            capacity_0=39500),
]
guarded = case
for b in BANDS:
    unguarded = unguarded.with_init(b, 0)
    guarded = guarded.with_init(b, 0)
tests.append(extreme("empty list, no referrals, no guard", unguarded,
                     referrals_0=0))
tests.append(extreme("empty list, no referrals, guarded", guarded,
                     referrals_0=0))
pd.set_option("display.max_colwidth", 90)
print(pd.DataFrame(tests).to_string(index=False))

Four of the six runs behave. Two do not.

* With an empty list and no guard, the model **crashes**: every treatment share is 0/0. The engine names the equation and the time (`Treating 0 gave NaN at time 0`). This is why the case model divides by `MAX(..., Smallest_list)`. An empty list is not exotic; a small specialty can clear its list.
* With urgent cases weighted 50 times and ten times the capacity, the floor holds `Waiting 0 to 3` at month 0.25. The share formula asks the first band for 37,289 patients a month while it holds 8,140, so one quarter-month step takes more than is there.

In [ ]:
r = case.run(capacity_0=0, include=["Waiting list", "Referrals"])
rise = r["Waiting list"].iloc[-1] - r["Waiting list"].iloc[0]
print(f"no capacity: the list rises by {rise:,.0f} patients; referrals "
      f"added up: {0.25 * r['Referrals'].iloc[:-1].sum():,.0f}")
r = case.run(priority=50, capacity_0=39500, warn_clips=False, include="all")
print("urgent first, clipped:", r.attrs["clipped"])
print(r.loc[[0, 0.25, 0.5], ["Waiting 0 to 3", "Treating 0", "Ageing 0"]]
      .round(1))

With no capacity, the list rises by 554,208 patients, exactly the referrals added up: a pass. The urgent-first failure is a real flaw, though a remote one. Shares protect the list as a whole, not each band, so a strong enough priority can overdraw a band. Try it 5 applies the fix from Level Protected by Flow (chapter 5): cap each band's treatments at what that band can supply.

**Parameter plausibility.** Every parameter should mean something you could measure, and its value should sit inside a range you can defend from evidence other than the fit. Writing the ranges down is itself a test: it forces you to say where each number comes from. The ranges below are illustrative; in a real study each would cite a source.

In [ ]:
plausible = pd.DataFrame([
    # name, low, high, unit, where the range comes from
    ("referrals_0", 3000, 5000, "patients/month", "GP referral counts"),
    ("referral_growth", 0.0, 0.05, "1/year", "population growth and ageing"),
    ("capacity_0", 3000, 5000, "patients/month", "clinic sessions x slots"),
    ("capacity_growth", -0.02, 0.05, "1/year", "staffing plans"),
    ("activity_at_worst", 0.2, 0.6, "dmnl", "activity reports, spring 2020"),
    ("recovery_time", 3, 18, "month", "activity reports, 2020 to 2021"),
    ("initiative_boost", 0.0, 0.2, "dmnl", "initiative budget"),
    ("priority", 1.0, 3.0, "dmnl", "clinical urgency rules"),
    ("minimum_treatment_time", 0.25, 1.0, "month", "booking lead time"),
], columns=["parameter", "low", "high", "unit", "evidence"]).set_index(
    "parameter")
plausible["value"] = [case.params[p] for p in plausible.index]
plausible["inside"] = plausible.value.between(plausible.low, plausible.high)
print(plausible.to_string())

All nine parameters sit inside their ranges. Two of the ranges matter later: section 7 samples from them, and chapter 10 uses them as bounds for calibration.

**Boundary adequacy.** A model's boundary is the line between what it explains and what it takes as given (chapter 2, From story to diagram). Here referrals are given. But when waits get long, some patients go private and some GPs refer less, so referrals may respond to the wait. The test is to add the missing loop and ask whether the *policy conclusion* changes. The wider model multiplies referrals by $(\text{expected wait} / 8\ \text{months})^{-0.3}$, where the expected wait is the list divided by treatments (months). To compare like with like, both versions get `Referrals 0` refitted to the data by a simple grid.

In [ ]:
loop = lb.Model.from_spec(waiting_list_spec(feedback=True))
print("units:", check_units(loop) or "no unit problems")


def fit_referrals(model):
    """The Referrals 0 (patients/month) that best fits the monthly totals."""
    grid = pd.DataFrame({"referrals_0": np.arange(3500, 5001, 5.0)})
    out = model.run_batch(grid, include=["Waiting list"])
    sims = out.pivot(index="time", columns="run", values="value")
    sims = sims.loc[np.arange(len(wl), dtype=float)].to_numpy()
    sse = ((sims - wl["total"].to_numpy()[:, None]) ** 2).sum(axis=0)
    return float(grid.referrals_0[np.argmin(sse)])


rows = []
for label, model in [("referrals fixed", case),
                     ("referrals respond to the wait", loop)]:
    r0 = fit_referrals(model)
    with_it = model.run(include=["Waiting list", "Expected wait"],
                        referrals_0=r0)
    without = model.run(include=["Waiting list"], referrals_0=r0,
                        initiative_boost=0)
    end = with_it["Waiting list"].iloc[-1]
    end0 = without["Waiting list"].iloc[-1]
    rows.append({"boundary": label, "referrals_0": r0,
                 "list Dec 2023": round(end),
                 "without initiative": round(end0),
                 "initiative effect": round(end - end0),
                 "effect (%)": round(100 * (end - end0) / end0, 1)})
print(pd.DataFrame(rows).to_string(index=False))

With referrals fixed, the initiative cuts the December 2023 list by 11,329 patients (14.1%). With referrals that respond to the wait, it cuts it by 6,013 (9.1%): shorter waits draw in more referrals, which take back part of the gain. The boundary matters for exactly the question a decision maker would ask, so the boundary test fails as it stands: we cannot claim the 14% until we know how strongly referrals respond. Section 6 brings in evidence from the data.

**Try it 5.** *Check:* in the no-capacity run, how many patients are in the 12-plus band at month 119? *Apply:* protect each treatment flow, writing `Treating 0` as `MIN(<its share>, Waiting_0_to_3 / Minimum_treatment_time)` and the same for the other bands, and rerun the urgent-first test. Does the fix change the normal run? *Extend:* repeat the boundary test with a wait elasticity of 0.1 and 0.6.

<details><summary>Worked answer</summary>

The next cell does all three. *Check:* 523,163 of the 586,484 patients, 89%. Nobody is treated, so everyone ages into the last band. *Apply:* the protected model passes; nothing is clipped. The normal run is unchanged (68,836 in December 2023), because the caps never bind there. But in the extreme run, 21,009 patients a month of capacity sits idle at month 0: the caps keep patients in the right bands, but the unused capacity is not passed on to other bands. A complete fix would share it out again. *Extend:* the initiative's effect is 12.6% with an elasticity of 0.1, 9.1% with 0.3 and 4.2% with 0.6. The conclusion depends strongly on a number nobody has measured.
</details>

In [ ]:
long_wait = case.run(capacity_0=0)[BANDS[3]].iloc[-1]
print(f"check: {long_wait:,.0f} patients waiting over 12 months")
protected = case
for i, band in enumerate(BANDS):
    share = case._find(f"Treating {i}")["eq"]
    key = band.replace(" ", "_")
    protected = protected.with_equation(
        f"Treating {i}", f"MIN({share}, {key} / Minimum_treatment_time)")
r = protected.run(priority=50, capacity_0=39500, warn_clips=False,
                  include="all")
used = r[[f"Treating {i}" for i in range(4)]].sum(axis=1)
print("apply: clipped", r.attrs["clipped"] or "nothing", "| idle at month 0:",
      round(r["Treatments"].iloc[0] - used.iloc[0]), "patients/month",
      "| normal run ends at",
      round(protected.run(include=["Waiting list"])["Waiting list"].iloc[-1]))
for e in [0.1, 0.3, 0.6]:
    m_e = loop.with_params(wait_elasticity=e)
    r0 = fit_referrals(m_e)
    a = m_e.run(referrals_0=r0, include=["Waiting list"]).iloc[-1, 0]
    b = m_e.run(referrals_0=r0, initiative_boost=0,
                include=["Waiting list"]).iloc[-1, 0]
    print(f"extend: elasticity {e}: effect {100 * (a - b) / b:.1f}%")

## 6. Validation: tests of behaviour

Behaviour tests compare the model's output with the **reference mode**, the pattern in the data that the model is meant to explain (chapter 2). Point-by-point fit comes in chapter 10. Here we compare **pattern measures**: a handful of numbers that describe the shape of the history, each chosen because it matters for the question. Four are enough to start: the growth rate before 2020, the rise in the year after the shock, the change after the initiative, and the month of the steepest rise.

In [ ]:
def patterns(total):
    """Pattern measures of a monthly list (a Series on the data's dates)."""
    v = total.to_numpy(dtype=float)
    return {
        "growth 2014 to 2019 (%/year)":
            100 * ((v[71] / v[0]) ** (12 / 71) - 1),
        "rise Mar 2020 to Feb 2021 (patients)": v[84] - v[73],
        "change 2022 to 2023 (patients)": v[119] - v[95],
        "month of the steepest rise": total.diff().idxmax().strftime("%b %Y"),
    }


model_bands = months(base)
check = pd.DataFrame({"data": patterns(wl.set_index("month")["total"]),
                      "model": patterns(model_bands["Waiting list"])})
print(check.to_string(float_format="{:,.1f}".format))
share_data = wl.w12_plus.iloc[-1] / wl.total.iloc[-1]
share_model = (model_bands["Waiting 12 plus"].iloc[-1]
               / model_bands["Waiting list"].iloc[-1])
print(f"share waiting over 12 months, Dec 2023: data {share_data:.0%}, "
      f"model {share_model:.0%}")

The model grows a little too fast before 2020 (5.9% a year against 5.3%), rises a little too little after the shock (17,488 against 19,223 patients) and, like the data, rises most steeply in March 2020. It also gets the mix right: 47% of the model's patients have waited over 12 months in December 2023, against 50% in the data. The figure shows the four bands.

In [ ]:
fig, ax = plt.subplots()
cols = ["w0_3", "w3_6", "w6_12", "w12_plus"]
for col, band in zip(cols, BANDS):
    line, = ax.plot(wl["month"], wl[col] / 1000, label=f"data {col}")
    ax.plot(wl["month"], model_bands[band] / 1000, color=line.get_color(),
            linestyle="", marker=".", markersize=3, label=f"model {band}")
ax.set(title="Patients by time waited, data and model",
       xlabel="month", ylabel="patients waiting (thousands)")
ax.legend(ncol=2, fontsize=8)
plt.show()

Figure: patients in each band of time waited, data (lines) and model (dots), January 2014 to December 2023, in thousands. Alt text: four pairs of lines; the 12-plus band rises steeply after 2020 and the model's dots follow each band closely.

The model reproduces the shape of all four bands, including the build-up in the 12-plus band after 2020, which it was not tuned to. A pattern the model was not built to reproduce is stronger evidence than one it was fitted to.

**Try it 6.** *Check:* what share of the list had waited over 12 months in December 2019, in the data and in the model? *Apply:* compute the pattern measures for the wider model of section 5 (referrals respond to the wait, `Referrals 0` refitted to 4,370 a month). *Extend:* which pattern measure tells the two models apart most clearly, and what does that say about the boundary test?

<details><summary>Worked answer</summary>

The next cell does it. *Check:* 39% in the data and 37% in the model. *Apply:* the wider model grows 8.1% a year before 2020, rises only 9,504 patients after the shock, and falls by 2,267 patients after the initiative where the data rose by 1,375. Its worst month is 23% off the data, against 4% for the case model. *Extend:* the rise after the shock. When treatments collapse, waits jump, and the wider model's referrals drop sharply, which halves the rise. The data show no such damping. With an elasticity of 0.3 the wider model fails the behaviour tests, so the evidence favours a weak response, and the 14% effect of the initiative is closer to the truth than the 9%. A boundary test and a behaviour test together settle what neither settles alone.
</details>

In [ ]:
i = 71                                                  # December 2019
share = model_bands[BANDS[3]].iloc[i] / model_bands["Waiting list"].iloc[i]
print(f"check: data {wl.w12_plus[i] / wl.total[i]:.0%}, model {share:.0%}")
wider = months(loop.run(referrals_0=4370))
print(pd.DataFrame({"data": patterns(wl.set_index("month")["total"]),
                    "case model": patterns(model_bands["Waiting list"]),
                    "wider model": patterns(wider["Waiting list"])})
      .to_string(float_format="{:,.1f}".format))
for label, tot in [("case", model_bands), ("wider", wider)]:
    off = (tot["Waiting list"].to_numpy() / wl["total"].to_numpy() - 1)
    print(f"{label} model, worst month: {np.abs(off).max():.0%} off")

## 7. Sensitivity: which assumptions carry the answer

Every parameter is uncertain. **Sensitivity analysis** varies them over their plausible ranges and asks which conclusions change. It helps to separate three kinds of sensitivity: *numerical* (the numbers move), *behavioural* (the shape of the behaviour changes) and *policy* (the preferred decision changes); the last is the one that matters most [@sterman2000].

**One at a time.** Set each parameter to the low and then the high end of its range, keeping the others at their base values, and record the outcome. Sorting by the swing gives a **tornado chart**.

In [ ]:
def outcomes(df):
    """The two numbers the decision maker asked about."""
    return {"list Dec 2023": df["Waiting list"].iloc[-1],
            "over 12 months, Dec 2023": df["Waiting 12 plus"].iloc[-1]}


base_out = outcomes(base)
tornado = []
for p in plausible.index:
    low = outcomes(case.run(**{p: plausible.low[p]}, warn_clips=False))
    high = outcomes(case.run(**{p: plausible.high[p]}, warn_clips=False))
    key, key12 = "list Dec 2023", "over 12 months, Dec 2023"
    tornado.append({"parameter": p,
                    "low": low[key] - base_out[key],
                    "high": high[key] - base_out[key],
                    "swing": abs(high[key] - low[key]),
                    "swing, over 12 months": abs(high[key12] - low[key12])})
tornado = pd.DataFrame(tornado).set_index("parameter").sort_values("swing")
print(f"base: list {base_out['list Dec 2023']:,.0f}, over 12 months "
      f"{base_out['over 12 months, Dec 2023']:,.0f} patients")
print(tornado.to_string(float_format="{:,.0f}".format))

In [ ]:
fig, ax = plt.subplots()
y = np.arange(len(tornado))
ax.barh(y, tornado.low / 1000, color="#56B4E9", edgecolor="black",
        label="parameter at its low end")
ax.barh(y, tornado.high / 1000, color="#E69F00", edgecolor="black",
        hatch="//", label="parameter at its high end")
ax.set_yticks(y, tornado.index)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title="One at a time: what moves the list in December 2023",
       xlabel="change in the list (thousand patients)", ylabel="")
ax.legend(loc="lower right")
plt.show()

Figure: the change in the December 2023 list when each parameter moves to the low (plain bars) or high (hatched bars) end of its range, in thousands of patients, with the widest swing at the top. Alt text: horizontal bars, long for the referral and capacity parameters and short for the rest.

The four referral and capacity parameters dwarf the rest: moving `referrals_0` across its range swings the list by 205,371 patients. Two parameters have no effect at all on the total. `minimum_treatment_time` never binds when capacity is short of demand. `priority` only moves patients between bands; it shifts the 12-plus band by 5,534 patients without changing the total. A parameter that does not matter for one outcome can matter for another, so the outcome must match the question.

**Monte Carlo.** One at a time misses combinations. A **Monte Carlo** analysis draws all parameters at once, here 4,000 times, uniformly from their ranges, and runs them all with `run_batch`, which steps every run together with numpy. **Standardised regression coefficients** (the slopes of a regression of the standardised outcome on the standardised inputs) rank the inputs.

In [ ]:
n = 4000
varied = plausible.index.drop("minimum_treatment_time")
draws = pd.DataFrame({p: rng.uniform(plausible.low[p], plausible.high[p], n)
                      for p in varied})
out = case.run_batch(draws, include=["Waiting list", "Waiting 12 plus"],
                     warn_clips=False)
lists = out[out.variable == "Waiting list"].pivot(
    index="time", columns="run", values="value")
monthly = lists.loc[np.arange(len(wl), dtype=float)]
final = monthly.iloc[-1]
print("list Dec 2023, 5th, 50th, 95th percentiles:",
      np.percentile(final, [5, 50, 95]).round(-2))
print("runs ending below the start (32,276):", int((final < 32276).sum()))
z = (draws - draws.mean()) / draws.std()
src, *_ = np.linalg.lstsq(np.c_[np.ones(n), z],
                          (final - final.mean()) / final.std(), rcond=None)
print("standardised regression coefficients (list, Dec 2023):")
print(pd.Series(src[1:], index=draws.columns).round(2)
      .sort_values(key=abs, ascending=False).to_string())

In [ ]:
q = np.percentile(monthly / 1000, [5, 25, 50, 75, 95], axis=1)
fig, ax = plt.subplots()
ax.fill_between(wl["month"], q[0], q[4], color="#56B4E9", alpha=0.35,
                label="90% of runs")
ax.fill_between(wl["month"], q[1], q[3], color="#0072B2", alpha=0.45,
                label="50% of runs")
ax.plot(wl["month"], q[2], color="black", label="median run")
ax.plot(wl["month"], wl["total"] / 1000, color="#D55E00", linestyle="--",
        label="data")
ax.set(title="4,000 runs over the plausible ranges",
       xlabel="month", ylabel="patients waiting (thousands)")
ax.legend(loc="upper left")
plt.show()

Figure: the spread of 4,000 runs with every parameter drawn from its range, January 2014 to December 2023: the middle 50% (dark band), the middle 90% (light band), the median (solid) and the data (dashed), in thousands of patients. Alt text: a fan that widens from a point in 2014 to a very wide band by 2023, with the data inside it.

The answer is useless: the 90% range for December 2023 runs from 1,900 to 271,100 patients, and 1,641 runs end below where the list started. The reason is structural. The list is the running total of a small difference between two large flows, referrals and capacity. Independent ranges of plus or minus 25% on each allow gaps from minus 1,971 to plus 1,958 patients a month, when the history says the gap is near zero.

That is a lesson about the inputs, not the model: they are not independent, and the data already constrain them. A crude fix keeps only runs that stay within 10% of the data in every month of 2014 to 2019.

In [ ]:
history = monthly.iloc[:72].to_numpy()
data_hist = wl["total"].to_numpy()[:72, None]
worst = np.abs(history / data_hist - 1).max(axis=0)
kept = worst < 0.10
print(f"runs within 10% of the data in every month of 2014 to 2019: "
      f"{kept.sum()} of {n}")
print("list Dec 2023 for those runs, 5th, 50th, 95th percentiles:",
      np.percentile(final[kept], [5, 50, 95]).round(-2))
gap = draws.referrals_0 - draws.capacity_0
print(f"referrals_0 - capacity_0 in kept runs: {gap[kept].min():.0f} to "
      f"{gap[kept].max():.0f} patients/month (all runs: {gap.min():.0f} to "
      f"{gap.max():.0f})")

Only 72 of 4,000 runs survive, and their referral-capacity gap lies between minus 148 and plus 197 patients a month. Their 90% range for December 2023 is 41,300 to 88,400 patients: still wide, but now about something. This filter is the first step towards calibration, and chapter 10 replaces it with proper estimation and uncertainty bands.

**Try it 7.** *Check:* which parameter has the largest swing for the 12-plus band in the tornado table? *Apply:* redo the Monte Carlo, but draw `capacity_0` as `referrals_0` minus 50, plus or minus 100 (uniform), so the gap is tied to the history. What is the 90% range now? *Extend:* compute the standardised regression coefficients for the 72 kept runs. What goes wrong?

<details><summary>Worked answer</summary>

The next cell does it. *Check:* `referrals_0`, with a swing of 143,901 patients. *Apply:* 2,000 to 179,400 patients, median 68,900. Tying the levels together narrows the range but not enough, because the two growth rates still vary independently and their gap compounds over ten years. *Extend:* the coefficients go above 2 in size (2.67 for `referrals_0` and minus 2.43 for `capacity_0`) with opposite signs. In the kept runs the two levels are strongly correlated, so the regression cannot separate their effects. Correlated inputs break simple sensitivity measures. Chapter 10 meets the same problem as correlated parameter estimates.
</details>

In [ ]:
print("check:", tornado["swing, over 12 months"].idxmax())
tied = draws.copy()
tied["capacity_0"] = tied.referrals_0 - 50 + rng.uniform(-100, 100, n)
out_t = case.run_batch(tied, include=["Waiting list"], warn_clips=False)
end_t = out_t.pivot(index="time", columns="run", values="value").iloc[-1]
print("apply: 5th, 50th, 95th percentiles",
      np.percentile(end_t, [5, 50, 95]).round(-2))
zk = (draws[kept] - draws[kept].mean()) / draws[kept].std()
fk = final[kept]
coef, *_ = np.linalg.lstsq(np.c_[np.ones(kept.sum()), zk],
                           (fk - fk.mean()) / fk.std(), rcond=None)
print("extend:")
print(pd.Series(coef[1:], index=draws.columns).round(2).to_string())

## 8. Tests that run themselves

Tests run once by hand are forgotten the next time the model changes. Writing each test as a small function that **asserts** what must be true turns this chapter into a safety net. The style is the one used by `pytest`, Python's standard test runner: a function whose name starts with `test_`, which raises an `AssertionError` with a helpful message when the check fails. Each function takes the model as an argument, so the same tests run on any version of the model.

In [ ]:
def test_units_agree(model):
    problems = check_units(model)
    assert not problems, problems[0]


def test_patients_are_conserved(model):
    gaps, clipped = balance_error(model)
    assert max(gaps.values()) < 1e-6, f"gap of {max(gaps.values()):.3g}"
    assert not clipped, f"floor held {list(clipped)}"


def test_halving_dt_changes_little(model):
    a = model.run(dt=0.25, include=["Waiting list"])["Waiting list"]
    b = model.run(dt=0.125, include=["Waiting list"])["Waiting list"]
    change = abs(a.iloc[-1] / b.iloc[-1] - 1)
    assert change < 0.01, f"{change:.1%} change"


def test_no_capacity_list_grows_by_referrals(model):
    df = model.run(capacity_0=0, include=["Waiting list", "Referrals"])
    added = 0.25 * df["Referrals"].iloc[:-1].sum()
    rise = df["Waiting list"].iloc[-1] - df["Waiting list"].iloc[0]
    assert abs(rise - added) < 1, f"rise {rise:.0f}, referrals {added:.0f}"


def test_no_referrals_list_empties_safely(model):
    df = model.run(referrals_0=0, warn_clips=False)
    assert df["Waiting list"].iloc[-1] < 1
    assert not df.attrs["clipped"], f"floor held {list(df.attrs['clipped'])}"


def test_an_empty_list_runs(model):
    for b in BANDS:
        model = model.with_init(b, 0)
    try:
        df = model.run(referrals_0=0, include=["Waiting list"])
    except lb.LoopBricksError as err:
        raise AssertionError(str(err)[:60])
    assert df["Waiting list"].max() == 0


def test_urgent_priority_never_overdraws_a_band(model):
    df = model.run(priority=50, capacity_0=39500, warn_clips=False)
    assert not df.attrs["clipped"], f"floor held {list(df.attrs['clipped'])}"


def test_parameters_plausible(model):
    out = [p for p in plausible.index
           if not plausible.low[p] <= model.params[p] <= plausible.high[p]]
    assert not out, f"outside the range: {out}"


def test_steepest_rise_in_march_2020(model):
    total = months(model.run(include=["Waiting list"]))["Waiting list"]
    when = total.diff().idxmax().strftime("%b %Y")
    assert when == "Mar 2020", when


def test_history_within_ten_percent(model):
    total = months(model.run(include=["Waiting list"]))["Waiting list"]
    worst = (total.to_numpy() / wl["total"].to_numpy() - 1).__abs__().max()
    assert worst < 0.10, f"worst month {worst:.1%} off"

A dozen lines collect the results into a **test sheet**: one row per test, one column per model version. We run it on the case model and on the version without the guard against an empty list.

In [ ]:
KIND = {"units_agree": "verification",
        "patients_are_conserved": "verification",
        "halving_dt_changes_little": "verification",
        "no_capacity_list_grows_by_referrals": "structure: extreme",
        "no_referrals_list_empties_safely": "structure: extreme",
        "an_empty_list_runs": "structure: extreme",
        "urgent_priority_never_overdraws_a_band": "structure: extreme",
        "parameters_plausible": "structure: parameters",
        "steepest_rise_in_march_2020": "behaviour",
        "history_within_ten_percent": "behaviour"}


def run_sheet(models):
    """Run every test_ function on every model; one row per test."""
    rows = []
    for name, fn in list(globals().items()):
        if not (name.startswith("test_") and callable(fn)):
            continue
        row = {"test": name[5:].replace("_", " "),
               "kind": KIND.get(name[5:], "")}
        for label, model in models.items():
            try:
                fn(model)
                row[label] = "pass"
            except AssertionError as err:
                row[label] = f"FAIL: {err}"
        rows.append(row)
    return pd.DataFrame(rows)


sheet = run_sheet({"case model": case, "no guard": lb.Model.from_spec(
    waiting_list_spec(guard=False))})
print(sheet.to_string(index=False))

The sheet tells the story of the chapter at a glance. Both versions pass verification and the behaviour tests. The unguarded version fails the empty-list test. Both fail the urgent-priority test, a known weakness worth recording rather than hiding: a sheet with a failure and a note on why it is acceptable for now is more honest than a sheet of passes.

To run the same tests outside the notebook, save the functions in a file such as `tests/test_waiting_list.py`, add a **fixture** that builds the model, and run `pytest` from a terminal:

```python
import pytest

@pytest.fixture
def model():
    return lb.Model.from_spec(waiting_list_spec())
```

`pytest` passes the fixture to every test that asks for an argument called `model`, and reports each pass and failure. Run it after every change to the model, before every result goes out.

**Try it 8.** *Check:* run the test sheet on the wider model of section 5 (`loop`). Which tests fail, and why? *Apply:* write `test_initiative_shortens_list(model)`, which asserts that the December 2023 list is shorter with the initiative than without it, and run it on the case model. *Extend:* the wider model fails the empty-list test even though it has the guard. Find the equation responsible.

<details><summary>Worked answer</summary>

The next cell does it. *Check:* three fail. The empty-list test (`Referrals gave NaN`), the urgent-priority test, as for the case model, and the history test, worst month 34.9% off, because the wider model was not refitted. *Apply:* the test passes. A test of the sign of a policy effect is cheap and catches gross errors in a policy run. *Extend:* `Referrals`. With an empty list the expected wait is 0 months, and $0^{-0.3}$ is infinite. The guard in `Weighted list` does not help; the new equation needs its own, for example a floor of a few days on the expected wait. Every new piece of structure needs the extreme tests again, which is exactly why the tests should run themselves.
</details>

In [ ]:
print(run_sheet({"wider model": loop}).to_string(index=False))


def test_initiative_shortens_list(model):
    run = model.run(include=["Waiting list"])
    without = model.run(include=["Waiting list"], initiative_boost=0)
    a, b = run.iloc[-1, 0], without.iloc[-1, 0]
    assert a < b, f"{a:,.0f} with the initiative, {b:,.0f} without"


test_initiative_shortens_list(case)
print("apply: test_initiative_shortens_list passes on the case model")

## Check it

Before any model result leaves your desk:

* **Units.** Run a unit check on every equation; name every conversion constant.
* **Books.** Check that every stock equals its flows added up, and read `df.attrs["clipped"]`. A clip is a failed test until you can explain it.
* **Time step.** Halve `dt`. If the answer that matters moves by more than you would care about, halve it again. Compare with `method="rk4"`, remembering that jumps rob RK4 of its accuracy.
* **Extremes.** Zero and huge for every input that a policy could push; an empty start for every stock.
* **Sheet.** Keep the tests as functions and run them after every change.

## Running case

This chapter put the waiting-list model through a full test sheet. It passes verification: units agree, patients are conserved and `dt = 0.25` months is accurate to about 0.4%. It reproduces the reference mode's main patterns, including the build-up of long waiters it was not tuned to. The tests found and fixed one fault (a crash on an empty list) and recorded one known weakness (a strong priority can overdraw the urgent band). They also exposed the open question that matters most for policy: whether referrals respond to the wait. The behaviour tests suggest the response is weak. Chapter 10 (Data, calibration and uncertainty) estimates the parameters from the data, measures the fit and puts uncertainty bands on the results.

## Tell the decision maker

"We have tested the model in four ways: the arithmetic, the logic under extreme conditions, its match to ten years of history, and how much the answers depend on uncertain inputs. It passes the tests that matter for the question of capacity. One assumption needs your attention: if long waits put people off being referred, extra capacity draws in new referrals and the benefit of the initiative shrinks, from about 14% to about 9% of the list. The history suggests the effect is small, but it has not been measured. A short audit of referral patterns would settle it."

## Going further

* **Formal testing frameworks.** @barlas1996 sets out a sequence of structure-oriented behaviour tests that go beyond those here, including tests that compare behaviour under modified structure. @martinezmoyano2013 lists testing practices from experienced modellers.
* **Loop dominance.** Knocking a loop out and watching what changes is a powerful structure test; chapter 6 (Why things swing) uses it.
* **Global sensitivity.** Variance-based methods (Sobol indices) handle interactions and nonlinearity better than regression coefficients. They need many runs, which `run_batch` makes cheap.
* **Continuous integration.** Services such as GitHub Actions run `pytest` on every change to a repository, so a broken model never reaches the main branch unnoticed. This book's notebooks are tested that way.

## Mini-project

Extend the test sheet into a full confidence report for the waiting-list model.

1. Add the protection of Try it 5, then share the idle capacity out to the other bands so that no capacity is wasted. Show that the urgent-priority test now passes and that patients are still conserved.
2. Add a guard to the wider model's `Referrals` and make it pass the empty-list test.
3. Add three tests of your own: one more extreme condition, one more pattern measure, one policy-direction test.
4. Present the final sheet with one sentence per test explaining why it matters for the capacity question.

## Further reading

* @barlas1996, the standard reference on validity and validation in system dynamics.
* @martinezmoyano2013, best practices in modelling, including testing.
* @sterman2000, chapter 21, on testing and building confidence in models.
* Hines, *Molecules of Structure* [@hines_molecules], for the four molecules used here. To try them in the browser, the LoopBricks app has [Reducing Backlog](https://waelrash1.github.io/loopbricks/#/m/reducingBacklog), [Level Protected by PDY](https://waelrash1.github.io/loopbricks/#/m/protByPDY), [Doing Work Cascade](https://waelrash1.github.io/loopbricks/#/m/doingWorkCascade) and [Cascade Protected by PDY](https://waelrash1.github.io/loopbricks/#/m/cascadeProtByPDY).